# Check of `counterfactual_simulation`
Independent re-computation of the merger equilibrium (no `blp` simulation helpers) compared with `blp.counterfactual_simulation`.

In [ ]:
using LinearAlgebra, Statistics, JLD2, Plots
include("blp.jl")
using .blp

# NB: the stored .jdl2 must match the current BLPDemand fields (ps_sim/ts_sim added later => re-estimate or re-save)
sys   = load("models/model_system_bert.jdl2")["model"]
Omega_merge = [1.0 1.0 0.0; 1.0 1.0 0.0; 0.0 0.0 1.0]

## Independent solver
Uses only `beta`, `xi`, `sigma`, `errors`, `X` and the pre-merger recovered costs. Market-by-market, damped fixed point on the Bertrand FOC, with the FOC residual as the stopping rule.

In [ ]:
function indep_shares_jac(d, m, p, Ω)
    J, R = d.J, d.R
    idx  = (m-1)*J+1:m*J
    βp   = d.beta[end]
    δnp  = d.X * d.beta[1:end-1] .+ d.xi          # mean utility excluding price
    α    = βp .- exp.(d.sigma .* d.errors[m, :])   # R draws of price coef (negative)
    s  = zeros(J); Δ = zeros(J, J)                 # Δ[j,k] = ds_j/dp_k
    for r in 1:R
        u  = δnp[idx] .+ α[r] .* p
        e  = exp.(u); sr = e ./ (1 + sum(e))
        s .+= sr ./ R
        Δ .+= α[r] .* (Diagonal(sr) - sr * sr') ./ R
    end
    return s, Δ
end

function indep_equilibrium(d, Ω, c; λ = 0.1, tol = 1e-10, maxit = 5_000)
    P = copy(d.p); res = zeros(d.T); iters = zeros(Int, d.T)
    for m in 1:d.T
        idx = (m-1)*d.J+1:m*d.J
        p = P[idx]; cm = c[idx]
        for it in 1:maxit
            s, Δ = indep_shares_jac(d, m, p, Ω)
            pnew = cm .- inv(Ω .* Δ) * s
            err  = maximum(abs.(pnew .- p))
            p    = λ .* pnew .+ (1-λ) .* p
            iters[m] = it
            err < tol && break
        end
        P[idx] = p
        s, Δ = indep_shares_jac(d, m, p, Ω)
        res[m] = maximum(abs.(p .- cm .+ inv(Ω .* Δ) * s))   # FOC residual
    end
    return P, res, iters
end

## 1. Baseline: pre-merger ownership must reproduce observed prices

In [ ]:
d0 = sys.BLPDemand
c0 = copy(d0.cost_sim)
P0, res0, it0 = indep_equilibrium(d0, d0.Omega, c0)
println("max |P_indep - p_observed| = ", maximum(abs.(P0 .- d0.p)), "   max FOC resid = ", maximum(res0))

## 2. Merger: module vs independent solver

In [ ]:
using Base: deepcopy
sim = deepcopy(sys)
sim.BLPDemand.Omega = Omega_merge
sim = blp.counterfactual_simulation(sim)   # needs p_old taken at the top of the loop, damping, abs-norm (see report)
P_mod = copy(sim.BLPDemand.p)

P_ind, res_ind, it_ind = indep_equilibrium(d0, Omega_merge, c0)

println("max |P_module - P_indep|    = ", maximum(abs.(P_mod .- P_ind)))
println("independent FOC residual    = ", maximum(res_ind), "  (iters max ", maximum(it_ind), ")")
println("max |P_merge - P_baseline|  = ", maximum(abs.(P_ind .- d0.p)))
# FOC residual of the module's solution, evaluated independently
res_mod = [ (idx = (m-1)*d0.J+1:m*d0.J; (s,Δ) = indep_shares_jac(d0, m, P_mod[idx], Omega_merge);
             maximum(abs.(P_mod[idx] .- c0[idx] .+ inv(Omega_merge .* Δ) * s))) for m in 1:d0.T ]
println("module FOC residual         = ", maximum(res_mod))

## 3. Average price change by product (expect 1 and 2 up, 3 slightly up)

In [ ]:
ΔP = reshape(P_ind .- d0.p, d0.J, d0.T)
println("mean price change by product (indep): ", round.(mean(ΔP, dims=2)[:], digits=4))
ΔPm = reshape(P_mod .- d0.p, d0.J, d0.T)
println("mean price change by product (module): ", round.(mean(ΔPm, dims=2)[:], digits=4))
scatter(P_ind, P_mod, xlabel="independent", ylabel="module", legend=false); plot!(x->x, color=:red)